# Pipeline Medallion — ANS Beneficiários SP

Versão notebook (estilo Databricks) do pipeline. Executa **Bronze → Silver → Gold** e as **3 consultas** do case.

> O mesmo código roda no Databricks: basta apontar `LAKE_ROOT` para o S3 real. Aqui usamos MinIO (S3 local) via Docker.

In [1]:
import sys
sys.path.append('/app')  # deixa achar os módulos src.* (código fica em /app)

from src.config import get_spark, sql_params   # sessão Spark + parâmetros dos .sql
from src.sql_runner import run_sql_file         # executa um arquivo .sql
from src.queries import run_queries             # roda as 3 consultas do case

spark = get_spark('case-ans-notebook')   # cria a sessão Spark (Delta + MinIO)
params = sql_params()                     # valores dos {{LAKE_ROOT}}/{{CSV_PATH}}
params

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b1461c86-4bb0-462e-83c0-81387c8dc78b;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 209ms :: artifacts dl 12ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 

{'LAKE_ROOT': 's3a://lakehouse',
 'CSV_PATH': '/app/data/pda-024-icb-SP-2025_08.csv'}

## Bronze — ingestão as-is
Carrega o CSV bruto para Delta, preservando a estrutura original.

In [2]:
run_sql_file(spark, '00_bronze.sql', params)  # ingere o CSV bruto -> Delta (as-is) create or replace bronze.beneficiarios
spark.table('bronze.beneficiarios').limit(5).toPandas()  # amostra de 5 linhas


=== 00_bronze.sql  (3 statements) ===
  [1/3] CREATE DATABASE IF NOT EXISTS bronze LOCATION 's3a://lakehouse/bronze'...
  [2/3] CREATE OR REPLACE TEMPORARY VIEW raw_csv USING csv OPTIONS ( path '/app/data/pda...
  [3/3] CREATE OR REPLACE TABLE bronze.beneficiarios USING delta AS SELECT *, current_ti...


,ID_CMPT_MOVEL,CD_OPERADORA,NM_RAZAO_SOCIAL,NR_CNPJ,MODALIDADE_OPERADORA,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO,TP_SEXO,DE_FAIXA_ETARIA,...,DE_SEGMENTACAO_PLANO,DE_ABRG_GEOGRAFICA_PLANO,COBERTURA_ASSIST_PLAN,TIPO_VINCULO,QT_BENEFICIARIO_ATIVO,QT_BENEFICIARIO_ADERIDO,QT_BENEFICIARIO_CANCELADO,DT_CARGA,_ingested_at,_source_file
0,2025-08,333051,UNIMED DE GUARULHOS COOPERATIVA DE TRABALHO MÉ...,74466137000172,COOPERATIVA MÉDICA,SP,351880,Guarulhos,F,20 a 24 anos,...,Ambulatorial + Hospitalar com obstetrícia,Municipal,Médico-hospitalar,Titular,1,0,0,2026-06-30,2026-08-05 06:12:14.418339,/app/data/pda-024-icb-SP-2025_08.csv
1,2025-08,350249,H.B. SAÚDE S/A.,02668512000156,MEDICINA DE GRUPO,SP,351120,Catiguá,M,30 a 34 anos,...,Ambulatorial + Hospitalar com obstetrícia,Grupo de municípios,Médico-hospitalar,Titular,1,0,0,2026-06-30,2026-08-05 06:12:14.418339,/app/data/pda-024-icb-SP-2025_08.csv
2,2025-08,364312,UNIMED DE ARARAQUARA - COOP. DE TRAB. MÉDICO,45272366000158,COOPERATIVA MÉDICA,SP,350320,Araraquara,F,65 a 69 anos,...,Ambulatorial + Hospitalar com obstetrícia,Grupo de municípios,Médico-hospitalar,Titular,1,0,0,2026-06-30,2026-08-05 06:12:14.418339,/app/data/pda-024-icb-SP-2025_08.csv
3,2025-08,346659,CAIXA DE ASSISTÊNCIA DOS FUNCIONÁRIOS DO BANCO...,33719485000127,AUTOGESTÃO,SP,354980,São José do Rio Preto,M,50 a 54 anos,...,Ambulatorial + Hospitalar com obstetrícia,Nacional,Médico-hospitalar,Titular,3,0,0,2026-06-30,2026-08-05 06:12:14.418339,/app/data/pda-024-icb-SP-2025_08.csv
4,2025-08,379956,CARE PLUS MEDICINA ASSISTENCIAL LTDA.,02725347000127,MEDICINA DE GRUPO,SP,352400,Itupeva,F,30 a 34 anos,...,Ambulatorial + Hospitalar com obstetrícia,Grupo de estados,Médico-hospitalar,Titular,2,0,0,2026-06-30,2026-08-05 06:12:14.418339,/app/data/pda-024-icb-SP-2025_08.csv


## Silver — tipagem + mascaramento
Tipa as colunas, mascara o CNPJ e particiona por competência.

In [3]:
run_sql_file(spark, '01_silver.sql', params)  # tipa, mascara CNPJ e particiona
spark.table('silver.beneficiarios').limit(5).toPandas()  # amostra de 5 linhas


=== 01_silver.sql  (2 statements) ===
  [1/2] CREATE DATABASE IF NOT EXISTS silver LOCATION 's3a://lakehouse/silver'...
  [2/2] CREATE OR REPLACE TABLE silver.beneficiarios USING delta PARTITIONED BY (id_cmpt...


,cd_operadora,nm_razao_social,nr_cnpj_masc,modalidade_operadora,sg_uf,cd_municipio,nm_municipio,tp_sexo,de_faixa_etaria,de_segmentacao_plano,de_abrg_geografica_plano,de_contratacao_plano,cobertura_assist_plan,tipo_vinculo,qt_beneficiario_ativo,qt_beneficiario_aderido,qt_beneficiario_cancelado,dt_carga,id_cmpt_movel
0,301949,ODONTOPREV S/A,58119199******,ODONTOLOGIA DE GRUPO,SP,353080,Mogi Mirim,M,25 a 29 anos,Odontológico,Nacional,Coletivo Empresarial,Odontológico,Dependente,1,0,0,2026-06-30,2025-08
1,368253,HAPVIDA ASSISTENCIA MEDICA S.A.,63554067******,MEDICINA DE GRUPO,SP,354460,Sabino,M,50 a 54 anos,Ambulatorial + Hospitalar com obstetrícia,Grupo de municípios,Individual ou Familiar,Médico-hospitalar,Titular,1,0,0,2026-06-30,2025-08
2,359017,NOTRE DAME INTERMÉDICA SAÚDE S.A.,44649812******,MEDICINA DE GRUPO,SP,355550,Ubirajara,F,30 a 34 anos,Ambulatorial + Hospitalar com obstetrícia,Nacional,Coletivo Empresarial,Médico-hospitalar,Titular,1,0,0,2026-06-30,2025-08
3,006246,SUL AMERICA COMPANHIA DE SEGURO SAÚDE,01685053******,SEGURADORA ESPECIALIZADA EM SAÚDE,SP,355030,São Paulo,F,65 a 69 anos,Ambulatorial + Hospitalar com obstetrícia,Nacional,Coletivo Empresarial,Médico-hospitalar,Titular,1,0,0,2026-06-30,2025-08
4,416801,UNIMED SAÚDE E ODONTO S.A,10414182******,MEDICINA DE GRUPO,SP,350460,Bady Bassitt,M,35 a 39 anos,Odontológico,Nacional,Coletivo Empresarial,Odontológico,Titular,1,0,0,2026-06-30,2025-08


## Gold — tabelas curadas
Agrega os dados para consumo analítico.

In [4]:
run_sql_file(spark, '02_gold.sql', params)  # agrega (1 tabela por pergunta)
spark.sql('SHOW TABLES IN gold').toPandas()  # lista as tabelas criadas


=== 02_gold.sql  (4 statements) ===
  [1/4] CREATE DATABASE IF NOT EXISTS gold LOCATION 's3a://lakehouse/gold'...
  [2/4] CREATE OR REPLACE TABLE gold.beneficiarios_por_operadora USING delta AS SELECT c...
  [3/4] CREATE OR REPLACE TABLE gold.beneficiarios_por_faixa_etaria USING delta AS SELEC...
  [4/4] CREATE OR REPLACE TABLE gold.beneficiarios_por_municipio USING delta AS SELECT c...


,namespace,tableName,isTemporary
0,gold,beneficiarios_por_faixa_etaria,False
1,gold,beneficiarios_por_municipio,False
2,gold,beneficiarios_por_operadora,False
3,,raw_csv,False


## Consultas do case
(a) top 5 operadoras — (b) faixa etária com mais beneficiários — (c) beneficiários por município.

In [5]:
run_queries(spark)  # (a) top 5 operadoras, (b) faixa etária, (c) municípios -> salva em output/


(a) Top 5 operadoras por beneficiários ativos:
+------------+------------------------------------------+-----------------------+
|cd_operadora|nm_razao_social                           |qt_beneficiarios_ativos|
+------------+------------------------------------------+-----------------------+
|359017      |NOTRE DAME INTERMÉDICA SAÚDE S.A.         |4324507                |
|301949      |ODONTOPREV S/A                            |2844728                |
|326305      |AMIL ASSISTÊNCIA MÉDICA INTERNACIONAL S.A.|2671950                |
|006246      |SUL AMERICA COMPANHIA DE SEGURO SAÚDE     |2111707                |
|000582      |PORTO SEGURO - SEGURO SAÚDE S/A           |1391580                |
+------------+------------------------------------------+-----------------------+


(b) Faixa etária com mais beneficiários:
+---------------+-----------------------+
|de_faixa_etaria|qt_beneficiarios_ativos|
+---------------+-----------------------+
|40 a 44 anos   |3100823                |
+--